# Pipeline B — Classic CV: Parameter Tuning Notebook

Use this notebook to test and tune each stage of the pipeline **one step at a time**.

**Workflow:**
1. Pick a test image (try both a Type 1 and Type 2)
2. Run each section, inspect the output
3. Adjust the parameters in the `# PARAMETERS` block of each section
4. Once happy, copy the final parameter values into the corresponding `.py` file

**Never** use this notebook as your submission code — it's a scratchpad only.

## 0. Setup & Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Make sure our pipeline modules are importable
sys.path.append('..')  # adjust if running from a different directory

from pipeline_B_classic.preprocessing import preprocess, visualize_steps as viz_pre
from pipeline_B_classic.segmentation import segment, visualize_segmentation as viz_seg
from pipeline_B_classic.postprocessing import postprocess, visualize_steps as viz_post
from pipeline_B_classic.skeletonization import skeletonize_mask, visualize_steps as viz_skel

# Helper: display any two masks side by side with a title
def compare(img_a, title_a, img_b, title_b, cmap='gray'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_a, cmap=cmap); axes[0].set_title(title_a); axes[0].axis('off')
    axes[1].imshow(img_b, cmap=cmap); axes[1].set_title(title_b); axes[1].axis('off')
    plt.tight_layout(); plt.show()

print('Imports OK')

ModuleNotFoundError: No module named 'pipeline_b_classic'

## 1. Load Test Images
Set the paths to one Type 1 and one Type 2 image here.
Run all sections below on both to compare behaviour.

In [ ]:
# ── CHANGE THESE PATHS ──────────────────────────────────────────────────────
TYPE1_PATH = '../data/raw/easy_001.png'   # side view, electrode base visible
TYPE2_PATH = '../data/raw/hard_001.png'   # top-down zoom, no electrode base
# ────────────────────────────────────────────────────────────────────────────

# Load raw for reference display only
raw_type1 = cv2.imread(TYPE1_PATH, cv2.IMREAD_GRAYSCALE)
raw_type2 = cv2.imread(TYPE2_PATH, cv2.IMREAD_GRAYSCALE)

assert raw_type1 is not None, f'Could not load: {TYPE1_PATH}'
assert raw_type2 is not None, f'Could not load: {TYPE2_PATH}'

compare(raw_type1, f'Type 1 — raw  {raw_type1.shape}',
        raw_type2, f'Type 2 — raw  {raw_type2.shape}')
print('Images loaded successfully')

## 2. Preprocessing
Tune: `clahe_clip`, `clahe_tile`, `bilateral_d`, `bilateral_sigma`

**What to look for:**
- Dendrite branches should be clearly brighter than background after CLAHE
- Bilateral filter output should look smoother in flat regions but edges still sharp
- The metadata bar at the bottom should be gone after cropping

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
CLAHE_CLIP    = 2.0   # try: 1.5, 2.0, 3.0
CLAHE_TILE    = 8     # try: 4, 8, 16
BILATERAL_D   = 9     # try: 7, 9, 11
BILATERAL_SIG = 4.5   # try: 3.0, 4.5, 6.0  (should be ~ diameter/2)
# ────────────────────────────────────────────────────────────────────────────

pre1 = preprocess(TYPE1_PATH, CLAHE_CLIP, CLAHE_TILE, BILATERAL_D, BILATERAL_SIG)
pre2 = preprocess(TYPE2_PATH, CLAHE_CLIP, CLAHE_TILE, BILATERAL_D, BILATERAL_SIG)

print('=== TYPE 1 ===')
viz_pre(pre1)

print('=== TYPE 2 ===')
viz_pre(pre2)

In [ ]:
# Histogram check — a good preprocessing output should show
# two rough humps: one for background (dark) one for dendrites (bright)
# If everything is bunched in one hump, CLAHE clip/tile needs adjustment

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, img, title in zip(axes,
                           [pre1['denoised'], pre2['denoised']],
                           ['Type 1 — denoised histogram', 'Type 2 — denoised histogram']):
    ax.hist(img.ravel(), bins=128, color='steelblue', edgecolor='none')
    ax.set_title(title)
    ax.set_xlabel('Pixel intensity')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Segmentation
Tune: `otsu_offset`, `adaptive_block_size`, `adaptive_C`, `method`

**What to look for:**
- Dendrite bodies should be white, background black
- Otsu mask should have clean dendrite shapes with minimal background noise
- Adaptive mask halo effect should be minimal (reduce block_size if halos visible)
- AND combination should be cleaner than either alone

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
OTSU_OFFSET         = 0    # shift Otsu threshold up/down: try -10, 0, +10
ADAPTIVE_BLOCK_SIZE = 11   # must be odd: try 11, 15, 21
ADAPTIVE_C          = 2    # subtracted from local mean: try 1, 2, 4
METHOD              = 'otsu'  # 'otsu', 'adaptive', or 'combined'
# ────────────────────────────────────────────────────────────────────────────

seg1 = segment(pre1['denoised'], OTSU_OFFSET, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C, METHOD)
seg2 = segment(pre2['denoised'], OTSU_OFFSET, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C, METHOD)

print('=== TYPE 1 ===')
viz_seg(seg1)

print('=== TYPE 2 ===')
viz_seg(seg2)

In [ ]:
# Quick comparison: overlay mask on original to check alignment
# Green pixels = classified as dendrite

def overlay_mask(gray_image, mask, alpha=0.4):
    """Overlay a green mask on top of a grayscale image."""
    rgb = cv2.cvtColor(gray_image, cv2.COLOR_GRAY2RGB)
    green = np.zeros_like(rgb)
    green[:, :, 1] = mask  # green channel only
    return cv2.addWeighted(rgb, 1.0, green, alpha, 0)

overlay1 = overlay_mask(pre1['denoised'], seg1['final_mask'])
overlay2 = overlay_mask(pre2['denoised'], seg2['final_mask'])

compare(overlay1, 'Type 1 — mask overlay', overlay2, 'Type 2 — mask overlay', cmap=None)

## 4. Postprocessing
Tune: `min_area`, `closing_kernel`, `erosion_size`

**What to look for:**
- `no_small`: noise speckles should disappear, main dendrite body intact
- `closed`: holes inside dendrite bodies should be filled
- `reconstructed`: thin branches should be recovered, isolated noise gone
- Final mask should look like a clean solid version of the segmentation output

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
MIN_AREA       = 300  # minimum pixel area to keep
CLOSING_KERNEL = -1   # -1 = adaptive (auto-detects Type 1 vs Type 2)
EROSION_SIZE   = 5    # reconstruction marker erosion kernel
ITERATIONS     = 3    # erosion iterations for marker creation
# ────────────────────────────────────────────────────────────────────────────

post1 = postprocess(seg1['final_mask'], MIN_AREA, CLOSING_KERNEL, EROSION_SIZE, ITERATIONS)
post2 = postprocess(seg2['final_mask'], MIN_AREA, CLOSING_KERNEL, EROSION_SIZE, ITERATIONS)

print('=== TYPE 1 ===')
viz_post(post1)

print('=== TYPE 2 ===')
viz_post(post2)

In [ ]:
# Before vs After postprocessing — the most important comparison
compare(seg1['final_mask'],       'Type 1 — before postprocessing',
        post1['reconstructed'],   'Type 1 — after postprocessing')

compare(seg2['final_mask'],       'Type 2 — before postprocessing',
        post2['reconstructed'],   'Type 2 — after postprocessing')

## 5. Full Pipeline Summary
Run the complete pipeline end-to-end and show:
`Raw → Preprocessed → Segmentation mask → Postprocessed mask`

This is the 4-panel figure format required for your submission visuals.

In [ ]:
def show_full_pipeline(image_path, pre_params, seg_params, post_params, title=''):
    """
    Shows the 4-stage pipeline summary for one image.
    Save this figure for your submission visuals.
    """
    raw  = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    pre  = preprocess(image_path, **pre_params)
    seg  = segment(pre['denoised'], **seg_params)
    post = postprocess(seg['final_mask'], **post_params)

    stages = [
        (raw,                    'Raw SEM Image'),
        (pre['denoised'],        'Preprocessed'),
        (seg['final_mask'],      'Segmentation Mask'),
        (post['reconstructed'],  'Postprocessed Mask'),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    fig.suptitle(title, fontsize=14)

    for ax, (img, stage_title) in zip(axes, stages):
        ax.imshow(img, cmap='gray')
        ax.set_title(stage_title, fontsize=12)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    return post['reconstructed']


# Use the best parameters you found above
pre_params  = dict(clahe_clip=CLAHE_CLIP, clahe_tile=CLAHE_TILE,
                   bilateral_d=BILATERAL_D, bilateral_sigma=BILATERAL_SIG)
seg_params  = dict(otsu_offset=OTSU_OFFSET, adaptive_block_size=ADAPTIVE_BLOCK_SIZE,
                   adaptive_C=ADAPTIVE_C, method=METHOD)
post_params = dict(min_area=MIN_AREA, closing_kernel=CLOSING_KERNEL,
                   erosion_size=EROSION_SIZE, iterations=ITERATIONS)

mask1 = show_full_pipeline(TYPE1_PATH, pre_params, seg_params, post_params, 'Type 1 — Full Pipeline')
mask2 = show_full_pipeline(TYPE2_PATH, pre_params, seg_params, post_params, 'Type 2 — Full Pipeline')

## 6. Skeletonization
Tune: `peak_min_distance`

**What to look for:**
- Distance transform: branch centers should be bright peaks, edges dark
- Watershed: each visually separate branch should be a different color
- Skeleton: single-pixel lines following the center of every branch
- Overlay: red tips at branch ends, blue forks at junctions — do they look correct?

In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────
PEAK_MIN_DISTANCE = 5   # min px between watershed seeds: try 3, 5, 8, 10
# ────────────────────────────────────────────────────────────────────────────

skel1 = skeletonize_mask(post1['reconstructed'], PEAK_MIN_DISTANCE)
skel2 = skeletonize_mask(post2['reconstructed'], PEAK_MIN_DISTANCE)

print('=== TYPE 1 ===')
a1 = skel1['analysis']
print(f'  Skeleton length : {a1["total_length"]} px')
print(f'  Tips (endpoints): {a1["tip_count"]}')
print(f'  Forks (junctions): {a1["fork_count"]}')
viz_skel(post1['reconstructed'], skel1)

print('=== TYPE 2 ===')
a2 = skel2['analysis']
print(f'  Skeleton length : {a2["total_length"]} px')
print(f'  Tips (endpoints): {a2["tip_count"]}')
print(f'  Forks (junctions): {a2["fork_count"]}')
viz_skel(post2['reconstructed'], skel2)

In [ ]:
# Full 5-panel submission figure:
# Raw → Preprocessed → Segmentation → Postprocessed → Skeleton
# This is the exact format required for your 5 submission visuals

def show_full_pipeline_with_skeleton(image_path, pre_params, seg_params,
                                      post_params, skel_params, title=''):
    raw  = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    pre  = preprocess(image_path, **pre_params)
    seg  = segment(pre['denoised'], **seg_params)
    post = postprocess(seg['final_mask'], **post_params)
    skel = skeletonize_mask(post['reconstructed'], **skel_params)

    # Build skeleton overlay (green=skeleton, red=tips, blue=forks)
    overlay = cv2.cvtColor(post['reconstructed'], cv2.COLOR_GRAY2RGB)
    overlay[skel['skeleton'] > 0]              = [0,   255, 0  ]
    overlay[skel['analysis']['tips'] > 0]      = [255, 0,   0  ]
    overlay[skel['analysis']['forks'] > 0]     = [0,   0,   255]

    stages = [
        (raw,                    'Raw SEM'),
        (pre['denoised'],        'Preprocessed'),
        (seg['final_mask'],      'Segmentation'),
        (post['reconstructed'],  'Postprocessed'),
        (overlay,                f'Skeleton\nTips={skel["analysis"]["tip_count"]} Forks={skel["analysis"]["fork_count"]}'),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(26, 6))
    fig.suptitle(title, fontsize=14)

    for ax, (img, stage_title) in zip(axes, stages):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(stage_title, fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'../outputs/visuals/{Path(image_path).stem}_full_pipeline.png',
                bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved to outputs/visuals/')


skel_params = dict(peak_min_distance=PEAK_MIN_DISTANCE)

show_full_pipeline_with_skeleton(TYPE1_PATH, pre_params, seg_params, post_params, skel_params,
                                  'Type 1 — Complete Pipeline')
show_full_pipeline_with_skeleton(TYPE2_PATH, pre_params, seg_params, post_params, skel_params,
                                  'Type 2 — Complete Pipeline')

## 7. Save Best Parameters
Once you're happy with the results, run this cell to print the final parameters.
Copy these values into the corresponding `.py` files as the new defaults.

In [ ]:
print('=' * 50)
print('FINAL PARAMETERS — copy these into your .py files')
print('=' * 50)
print(f'\n# preprocessing.py defaults:')
print(f'clahe_clip    = {CLAHE_CLIP}')
print(f'clahe_tile    = {CLAHE_TILE}')
print(f'bilateral_d   = {BILATERAL_D}')
print(f'bilateral_sig = {BILATERAL_SIG}')
print(f'\n# segmentation.py defaults:')
print(f'otsu_offset         = {OTSU_OFFSET}')
print(f'adaptive_block_size = {ADAPTIVE_BLOCK_SIZE}')
print(f'adaptive_C          = {ADAPTIVE_C}')
print(f'method              = "{METHOD}"')
print(f'\n# postprocessing.py defaults:')
print(f'min_area       = {MIN_AREA}')
print(f'closing_kernel = {CLOSING_KERNEL}')
print(f'erosion_size   = {EROSION_SIZE}')
print(f'iterations     = {ITERATIONS}')
print(f'\n# skeletonization.py defaults:')
print(f'peak_min_distance = {PEAK_MIN_DISTANCE}')

## 8. Save Output Masks & Skeletons
Save the final masks and skeletons to `outputs/` for use in evaluation and the report.

In [ ]:
from pathlib import Path

mask_dir  = Path('../outputs/masks');     mask_dir.mkdir(parents=True, exist_ok=True)
skel_dir  = Path('../outputs/skeletons'); skel_dir.mkdir(parents=True, exist_ok=True)

stem1 = Path(TYPE1_PATH).stem
stem2 = Path(TYPE2_PATH).stem

cv2.imwrite(str(mask_dir / f'{stem1}_mask.png'),     mask1)
cv2.imwrite(str(mask_dir / f'{stem2}_mask.png'),     mask2)
cv2.imwrite(str(skel_dir / f'{stem1}_skeleton.png'), skel1['skeleton'])
cv2.imwrite(str(skel_dir / f'{stem2}_skeleton.png'), skel2['skeleton'])

print(f'Masks saved to:     {mask_dir.resolve()}')
print(f'Skeletons saved to: {skel_dir.resolve()}')
print(f'\nType 1 — tips: {skel1["analysis"]["tip_count"]}, forks: {skel1["analysis"]["fork_count"]}, length: {skel1["analysis"]["total_length"]}px')
print(f'Type 2 — tips: {skel2["analysis"]["tip_count"]}, forks: {skel2["analysis"]["fork_count"]}, length: {skel2["analysis"]["total_length"]}px')